In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del archivo genre.csv

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/genre.csv")
)

genreId,genreName
12,Adventure
14,Fantasy
16,Animation
18,Drama
27,Horror
28,Action
35,Comedy
36,History
37,Western
53,Thriller


In [0]:
genre_schema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
])
genre_df = (
    spark.read
        .format('csv')
        .schema(genre_schema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/genre.csv")
)

genreId,genreName
null,genreName
12,Adventure
14,Fantasy
16,Animation
18,Drama
27,Horror
28,Action
35,Comedy
36,History
37,Western


root
 |-- genreId: integer (nullable = true)
 |-- genreName: string (nullable = true)



In [0]:
genre_df_final = (
    add_ingestion_date(genre_df)
    .select(
        col("genreId").alias("genre_id"),
        col("genreName").alias("genre_name"),
        lit(v_env).alias("env"),
        lit(v_file_date).alias("file_date")
    )
)

genre_id,genre_name,env,file_date
null,genreName,dev,2024-12-16
12,Adventure,dev,2024-12-16
14,Fantasy,dev,2024-12-16
16,Animation,dev,2024-12-16
18,Drama,dev,2024-12-16
27,Horror,dev,2024-12-16
28,Action,dev,2024-12-16
35,Comedy,dev,2024-12-16
36,History,dev,2024-12-16
37,Western,dev,2024-12-16


In [0]:
(
    genre_df_final.write
        .mode("overwrite")
        .saveAsTable("movie_silver.genres")
)

In [0]:
dbutils.notebook.exit("Success")